# Summary & Outlook

## Summary

Here are some important things to remember for your future work with CUDA:

### General GPU Code Design

CUDA code requires dedicated GPU kernel functions which are executed asynchronously, therefore, kernels must be of return type `void` and explicit synchronization is required:

```c++
__global__ void kernel() {
    // ...
}

int main(int argc, char *argv[]) {
    // ...
    kernel<<<numBlocks, numThreads>>>();
    cudaDeviceSynchronize();
    // ...
}
```

### Parallelism on the GPU

CUDA uses a thread hierarchy, which is based on the design of a GPU:
- All threads that execute a kernel are organized in a __grid__, which runs on the GPU
- The __grid__ is organized in __blocks__, each block is assigned to one Streaming Multiprocessor (SM), the smallest self-sustaining unit of the GPU
- Each __block__ holds the same number of __threads__
- The SM will execute a fixed number of threads (usually 32) of a single block simultaneously in a __warp__

The following built-in variables can be used to interact with this hierarchy:
- The number of blocks in a grid: `gridDim.x`
- The number of threads in a block: `blockDim.x`
- The index of the block within the grid: `blockIdx.x`
- The index of a thread within the local block: `threadIdx.x`

To calculate the __global thread index__: `blockIdx.x * blockDim.x + threadIdx.x`
To calculate the __total number of threads__ in the grid: `blockDim.x * gridDim.x`

With this, a __Grid-Stride Loop__ can be designed to compute an arbitrary number of output elements:

```c++
__global__ void kernel() {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    
    for (int i = start; i < 10; i += stride)
        // ...
}

int main(int argc, char *argv[]) {
    // ...
    kernel<<<numBlocks, numThreads>>>();
    cudaDeviceSynchronize();
    // ...
}
```

Choose a number of blocks that is a multiple of the number of SMs of the GPU.
Query the number of SMs of the GPU with:
```c++
int device;
int numberOfSMs;

cudaGetDevice(&deviceId);
cudaDeviceGetAttribute(&numberOfSMs, cudaDevAttrMultiProcessorCount, deviceId);
```

Choose a number of threads that is a multiple of the warp size of 32.

Unfortunately, there is no "silver bullet" for a grid configuration that always produces good performance.
Instead, each kernel may require a different configuration, in some cases, even depending on the GPU architecture.
To find a good configuration, test different parameter combinations and use profilers, such as Nsight Systems!

Limits apply, the the [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/compute-capabilities.html#compute-capabilities-table-device-and-streaming-multiprocessor-sm-information-per-compute-capability).

### Memory Management

GPUs and CPUs have different memory spaces, therefore, measures must be taken to enable data transfers between Host (CPU) and and Device (GPU).
Two different concepts were discussed in this course:

#### Managed Memory

Managed Memory is automatically transferred between the host and device.
Memory has to be allocated with
```c++
cudaMallocManaged(void** ptr_to_unified_ptr, size_t size_in_bytes)
```
and released with
```c++
cudaFree(void* unified_ptr)
```

Managed memory uses a page fault mechanic to transfer pages to the destination.
This usually results in poor performance.
To accelerate memory transfers, prefetching can be used, to the GPU:
```c++
int device;
cudaGetDevice(&device);

checkCudaError(cudaMemPrefetchAsync(data, numElements * sizeof(double), { .type = cudaMemLocationTypeDevice, .id = device }, 0));
```
To the CPU:
```c++
checkCudaError(cudaMemPrefetchAsync(data, numElements * sizeof(double), { .type = cudaMemLocationTypeHost }, 0));
```

#### Explicit CUDA Memory APIs

When using explicit memory handling APIs, dedicated buffers must be used for the host and device with explicit data transfer functions.
To allocate a buffer on the GPU, use:
```c++
cudaMalloc(void** ptr_to_device_ptr, size_t size_in_bytes)
```
To release the buffer, use:
```c++
cudaFree(void* device_ptr)
```
To allocate a buffer on the host with pinned memory (cannot be swapped):
```c++
cudaMallocHost(void** ptr_to_host_ptr, size_t size_in_bytes)
```
To release the buffer, use:
```c++
cudaFreeHost(void* host_ptr)
```
To transfer memory, use:
```c++
cudaMemcpy(void* destination, const void* source, size_t size_in_bytes, cudaMemcpyKind transfer_direction)
```
The `transfer_direction` may be:
  * `cudaMemcpyHostToDevice`,
  * `cudaMemcpyDeviceToHost`,
  * `cudaMemcpyHostToHost`,
  * `cudaMemcpyDeviceToDevice`, or
  * `cudaMemcpyDefault`.

### Shared Memory

Parts of the L1 cache of a SM can be used as shared memory.
This shared memory is shared between all threads within a block, but not beyond!
Shared memory can be useful for, e.g., reduction codes.

We discussed the allocation of compile-time constant buffers in the kernel with:
```c++
__global__ void kernel() {
    // ...
    // allocate shared memory
    __shared__ int shared_data[256];
    // ...
}
```
An alternative method for dynamic-sized shared memory buffers with a size known at kernel launch time is discussed in the [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/cpp-language-extensions.html#shared-memory).

To synchronize threads within the same block, use:
```c++
__syncthreads()
```
Synchronization across block boundaries is done via separate kernel launches, there are no dedicated functions for that purpose.

### Reductions

A reduction in CUDA can be implemented manually, using shared memory and thread-safe, atomic functions.
However, a more effective method is available by using the `cub` library in the kernel, e.g.:
```c++
// specialize the type with <accumulator data type, number of threads per block>
using BlockReduce = cub::BlockReduce<int, 256>;

// allocate shared memory for the block reduction
__shared__ typename BlockReduce::TempStorage tempStorage;

// perform the block reduction with the `Sum` operation
int block_acc = BlockReduce(tempStorage).Sum(thread_acc)
```

## Outlook

Good job on completing this course!
To continue learning about CUDA and GPU programming, you can check out the following resources:

* [CUDA C++ Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/index.html)
* [CUDA Samples](https://github.com/NVIDIA/cuda-samples)

NHR@FAU also offers additional courses on GPU programming - the recommended follow-up course after this one being **Scaling CUDA Applications**.
Upcoming dates for this and all other courses are listed on [go-nhr.de/trainings](https://go-nhr.de/trainings).